# Lab: Rayleigh distillation and the isotope thermometer

```{note}
This lab accompanies {doc}`paleoclimate`. The Rayleigh distillation model sketched there in a few lines of algebra is built here in a few lines of numpy, following the calculation Willi Dansgaard first published in the 1950s.
```

Ice cores are read as thermometers because the isotopic composition of snow depends on the temperature at which it condensed. In this lab you will reproduce that dependence from first principles. By the end you should be able to

- convert between isotope ratios and delta notation relative to the VSMOW standard,
- derive and implement the Rayleigh distillation law for a constant fractionation factor,
- connect the fraction of vapor remaining in an air parcel to its temperature through the saturation vapor pressure,
- predict $\delta^{18}\mathrm{O}$ of precipitation as a parcel cools from 25 °C to $-30$ °C and extract the slope $d\delta/dT$,
- compare the model slope with the observed spatial slope and reason about why spatial and temporal calibrations differ.

Everything runs on numpy and matplotlib alone.


## Delta notation and the standard

The quantity of interest is the ratio of heavy to light oxygen in a water sample, $R = {}^{18}\mathrm{O}/{}^{16}\mathrm{O}$, a number near $0.002$ everywhere on Earth. Measuring $R$ absolutely is very hard, but measuring the small difference between a sample and a standard is comparatively easy, which is why isotope geochemistry runs on delta notation,

$$ \delta^{18}\mathrm{O} = \left(\frac{R_{\mathrm{sample}}}{R_{\mathrm{VSMOW}}} - 1\right)\times 1000\ \text{‰}, $$

where the standard is Vienna Standard Mean Ocean Water and the factor of 1000 puts the answer in per mil. Average ocean water has $\delta^{18}\mathrm{O} \approx 0$ ‰ by construction, which means a composition matching the standard rather than an absence of heavy oxygen. Polar snow runs from about $-20$ ‰ on coastal Greenland to below $-55$ ‰ on the East Antarctic plateau, and the whole job of this lab is to explain those numbers.

**Task 1:** Complete the two conversion functions below, then verify a worked example. A sample with $R = 0.00193$ measured against a standard with $R_{\mathrm{VSMOW}} = 0.00200$ should give $\delta^{18}\mathrm{O} = -35$ ‰. Check that your two functions are inverses by converting a value out and back.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

R_VSMOW = 2.0052e-3   # the 18O/16O ratio of the VSMOW standard


def delta_from_R(R, R_std=R_VSMOW):
    """Delta value in per mil from an isotope ratio R."""
    # YOUR CODE HERE
    raise NotImplementedError


def R_from_delta(delta, R_std=R_VSMOW):
    """Isotope ratio R from a delta value in per mil."""
    # YOUR CODE HERE
    raise NotImplementedError


# Worked example from the slides: should print -35.0
# print(delta_from_R(0.00193, R_std=0.00200))

# Round trip: should print -35.0 again
# print(delta_from_R(R_from_delta(-35.0, R_std=0.00200), R_std=0.00200))


## How much vapor is left

Rayleigh distillation tracks an air parcel that cools as it travels from a warm ocean toward an ice sheet, condensing moisture and dropping it as precipitation along the way. The bookkeeping variable is $f$, the fraction of the original vapor still aboard. If the parcel always holds exactly its saturation load, which is the standard assumption of relative humidity pinned at 100%, then $f$ follows directly from the saturation vapor pressure,

$$ f(T) = \frac{e_s(T)}{e_s(T_i)}, $$

where $T_i$ is the temperature at which the parcel left the ocean. For $e_s$ we use the Magnus approximation to the Clausius-Clapeyron relation,

$$ e_s(T) = 6.1094\,\exp\!\left(\frac{17.625\,T}{T + 243.04}\right), $$

which returns hectopascals when $T$ is in degrees Celsius.

```{note}
The course handout this lab descends from labels $T$ in the Magnus formula as Kelvin. The constants 17.625 and 243.04 belong to the Celsius form, and the formula only behaves with Celsius input. A quick check settles it, since $e_s(0\,^\circ\mathrm{C})$ should be about 6.11 hPa, the vapor pressure near the triple point, and that is exactly what the formula gives at $T = 0$. Keep your temperatures in Celsius throughout this lab.
```

**Task 2:** Implement `e_sat` and `vapor_fraction` below. Sanity-check that $e_s(0) \approx 6.11$ hPa and $e_s(25) \approx 31.7$ hPa, then plot both $e_s(T)$ and $f(T)$ as the parcel cools from $T_i = 25$ °C down to $-30$ °C. How much of the original moisture survives to 0 °C? To $-30$ °C?


In [ ]:
T_i = 25.0   # source temperature, deg C


def e_sat(T):
    """Saturation vapor pressure in hPa; T in deg C (Magnus form)."""
    # YOUR CODE HERE
    raise NotImplementedError


def vapor_fraction(T, Ti=T_i):
    """Fraction of original vapor remaining at temperature T."""
    # YOUR CODE HERE
    raise NotImplementedError


T = np.arange(25.0, -30.0 - 0.5, -0.5)   # cooling path, fine increments

# print(f'e_s(0)  = {e_sat(0.0):.2f} hPa  (expect ~6.11)')
# print(f'e_s(25) = {e_sat(25.0):.2f} hPa (expect ~31.7)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# ax1.plot(T, e_sat(T), lw=2)
ax1.set_xlabel('temperature (deg C)')
ax1.set_ylabel('saturation vapor pressure (hPa)')
# ax2.plot(T, vapor_fraction(T), lw=2)
ax2.set_xlabel('temperature (deg C)')
ax2.set_ylabel('fraction of vapor remaining, f')
plt.tight_layout()
plt.show()


## The distillation law in delta form

Each parcel of condensate that forms is enriched in heavy oxygen relative to the vapor by the fractionation factor $\alpha = R_{\mathrm{liquid}}/R_{\mathrm{vapor}}$, slightly greater than one because the heavy molecule prefers the condensed phase. Following the chapter, removing condensate in the ratio $\alpha R_v$ drives $d(\ln R_v) = (\alpha - 1)\, d(\ln f)$, and for constant $\alpha$ the integral is the Rayleigh distillation law,

$$ \frac{R_v}{R_{v,0}} = f^{\,\alpha - 1}. $$

We adopt $\alpha = 1.009$, a representative value for oxygen near 0 °C. To work in delta notation, substitute $R = R_{\mathrm{VSMOW}}\,(\delta/1000 + 1)$ into both sides. The standard cancels, leaving

$$ \frac{\delta_v/1000 + 1}{\delta_{v,0}/1000 + 1} = f^{\,\alpha - 1}, $$

where the factors of 1000 are needed because our deltas are in per mil. Two more pieces complete the model. The initial vapor formed in equilibrium with ocean water of $\delta^{18}\mathrm{O} = 0$ ‰, so the vapor starts out depleted,

$$ \delta_{v,0} = \left(\frac{1}{\alpha} - 1\right)\times 1000 \approx -8.9\ \text{‰}, $$

and the precipitation forming at any moment sits one fractionation step above the vapor it leaves,

$$ \delta_p = \left[\left(\frac{\delta_v}{1000} + 1\right)\alpha - 1\right]\times 1000. $$

**Task 3:** Implement `delta_vapor` and `delta_precip` below. Make the two classic graphs, $\delta^{18}\mathrm{O}$ of vapor and precipitation against $f$, and the same two curves against $T$. Confirm that the precipitation curve starts near 0 ‰ at 25 °C and reaches a few tens of per mil below zero by $-30$ °C, in the range of real polar snow.


In [ ]:
alpha = 1.009
delta_v0 = (1.0 / alpha - 1.0) * 1000.0   # initial vapor, per mil
print(f'initial vapor delta18O = {delta_v0:.2f} per mil')


def delta_vapor(f, a=alpha, dv0=delta_v0):
    """delta18O of the remaining vapor (per mil) at vapor fraction f."""
    # YOUR CODE HERE
    raise NotImplementedError


def delta_precip(f, a=alpha, dv0=delta_v0):
    """delta18O of precipitation (per mil) forming at vapor fraction f."""
    # YOUR CODE HERE
    raise NotImplementedError


# f = vapor_fraction(T)
# dv, dp = delta_vapor(f), delta_precip(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# ax1.plot(f, dv, lw=2, label='vapor')
# ax1.plot(f, dp, lw=2, label='precipitation')
ax1.set_xlabel('fraction of vapor remaining, f')
ax1.set_ylabel('delta 18O (per mil)')
ax1.legend()
# ax2.plot(T, dv, lw=2, label='vapor')
# ax2.plot(T, dp, lw=2, label='precipitation')
ax2.set_xlabel('temperature (deg C)')
ax2.set_ylabel('delta 18O (per mil)')
ax2.legend()
plt.tight_layout()
plt.show()


## The slope of the thermometer

The thermometer is the slope of the precipitation curve, $d\delta_p/dT$. Observations across Greenland and Antarctica give a strikingly linear spatial relation between mean annual $\delta^{18}\mathrm{O}$ of snow and local temperature, with a slope near 0.6 to 0.7 ‰ per °C, the calibrations of Dansgaard (1964), Lorius and Merlivat (1977), and Johnsen and colleagues (1989) discussed in the chapter.

**Task 4:** Compute the local slope of your modeled $\delta_p(T)$ curve with `np.gradient`, plot it against temperature, and shade the observed 0.6 to 0.7 ‰ per °C band for comparison. Also print the average slope over the full path from 25 °C to $-30$ °C and over the polar end of the path from $-10$ °C to $-30$ °C. Where does the constant-$\alpha$ model land relative to the observations, and does the slope steepen or flatten as the parcel gets colder?


In [ ]:
# slope = np.gradient(dp, T)   # per mil per deg C

fig, ax = plt.subplots(figsize=(6, 4))
# ax.plot(T, slope, lw=2, label='Rayleigh model, alpha = 1.009')
ax.axhspan(0.6, 0.7, color='orange', alpha=0.3, label='observed spatial slope')
ax.set_xlabel('temperature (deg C)')
ax.set_ylabel('d(delta 18O)/dT (per mil per deg C)')
ax.legend()
plt.show()

# YOUR CODE HERE: print the average slope from 25 to -30 C, and from
# -10 to -30 C (a simple difference of endpoints divided by delta T works).


## What sets the calibration

The slope you just computed is not a constant of nature. It depends on the fractionation factor, which in reality grows as the air cools, and on the temperature of the moisture source, which moves with climate. Both knobs are one line of code away.

**Task 5:** Recompute the precipitation curve for $\alpha = 1.006, 1.009,$ and $1.012$ with the source held at 25 °C, and then for source temperatures $T_i = 15, 20, 25,$ and $30$ °C with $\alpha$ held at 1.009. Plot the two families of $\delta_p(T)$ curves side by side. Notice that changing the source temperature shifts the value of $\delta_p$ delivered to a polar site even when the site temperature never changes, which is the seed of the calibration problem discussed next.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for a in [1.006, 1.009, 1.012]:
    dv0 = (1.0 / a - 1.0) * 1000.0
    # YOUR CODE HERE: compute delta_precip along T with this alpha
    # ax1.plot(T, dp_a, lw=2, label=f'alpha = {a}')
    pass
ax1.set_xlabel('temperature (deg C)')
ax1.set_ylabel('delta 18O of precipitation (per mil)')
ax1.legend()

for Ti in [15.0, 20.0, 25.0, 30.0]:
    Tpath = np.arange(Ti, -30.0 - 0.5, -0.5)
    # YOUR CODE HERE: compute f along Tpath with this source temperature,
    # then delta_precip, and plot against Tpath
    # ax2.plot(Tpath, dp_Ti, lw=2, label=f'T_i = {Ti:.0f} C')
    pass
ax2.set_xlabel('temperature (deg C)')
ax2.legend()
plt.tight_layout()
plt.show()


## Synthesis: spatial versus temporal calibration

The Rayleigh model reproduces the modern spatial relation between snow isotopes and site temperature remarkably well for something this simple. Using it to read climate history asks for more. Applying the modern spatial slope to one site through time assumes the whole distillation path shifted the way the modern map varies from place to place, and your Task 5 plots show how easily that assumption can fail.

Read Jouzel et al. (1997), *Validity of the temperature reconstruction from water isotopes in ice cores*, and write a short paragraph addressing the following.

1. Summarize three processes from the paper that make the simple Rayleigh equation inadequate when applied to ice cores. Candidates you have now seen in miniature include changes in the moisture source, changes in the seasonality of snowfall, and the difference between surface temperature and the condensation temperature above the polar inversion.
2. In central Greenland the borehole-temperature calibration showed the spatial slope underestimates the glacial-interglacial temperature change by nearly a factor of two. Which of the processes in your list could produce an error of that sign?
3. Jouzel and colleagues nevertheless argue the isotope thermometer works rather well. Summarize their reasoning, and state in one sentence what additional measurement turns an isotope record into a calibrated temperature record.
